# Text Labeling, Classification & Local LLM Workflows  
## *IGF Code* (maritime gas-fuel rules) — weak labels, baselines, Ollama, EN→KO translation & PDF

This hands-on workshop builds **training data** and **text classifiers** from the maritime regulatory file **`data/IGF Code (124Pages).pdf`**: raw PDF text → cleaned, traceable chunks → taxonomy and **weak labels** → simulated human review and `gold_label` → **TF–IDF + logistic regression** with metrics and error analysis.

**Section A** runs entirely **without** any large language model. **Sections B and C** call **[Ollama](https://ollama.com)** so you run models **locally**—after you **pull** model weights once from the Ollama library, Python talks to **`http://localhost:11434`** (same tags as in `ollama run` / `ollama pull`).

---

### Ollama: install, pull, and match tags (before Section B or C)

1. **Install** the Ollama application for your OS from [ollama.com](https://ollama.com). Leave it running so the API is available (desktop app or `ollama serve`).
2. **Pull** the chat model you plan to use. Downloads are large (often multi‑GB). In **Terminal / PowerShell / Command Prompt**:
   - Default tag used in this notebook: **`ollama pull llama3`** (Meta **Llama 3**, alias tag `llama3`).
   - For other sizes or families, pick a tag from the [model library](https://ollama.com/library), then run **`ollama pull <tag>`** (examples: `llama3.1:8b`, `llama3.2:3b`, `qwen2.5:7b`).
3. **Verify** the model is present: **`ollama list`** shows your tag with a size. Optional smoke test: **`ollama run llama3`** (or your tag), ask a short question, then exit.
4. **Align with the notebook:** set **`OLLAMA_MODEL`** (Section B, cell **B0.1**) and **`OLLAMA_TRANSLATION_MODEL`** (Section C) to **exactly** the same strings as in **`ollama list`**. If you see errors like *model not found*, run **`ollama pull`** for that tag first.
5. **Hardware:** local inference uses **RAM** and optionally **GPU VRAM**. Smaller tags (e.g. `llama3.2:3b`) need less memory; close heavy apps if you hit slowdowns or crashes.

---

### How this notebook is organized

| Part | Focus |
|------|--------|
| **Section A** | **Classical ML (no LLM):** extract & clean chunks → taxonomy → weak labels → simulated review → TF–IDF + logistic regression → metrics and interpretability. |
| **Section B** | **Local LLM via Ollama:** 15 patterns (classification, rationales, batch labeling, extraction, …). **Requires** a **pulled** model matching **`OLLAMA_MODEL`**. |
| **Section C** | **English → Korean:** configurable **`OLLAMA_TRANSLATION_MODEL`**, preview, UTF‑8 `.txt`, PDF export. **Pull** that tag before running translation cells. |

**Suggested order (Section A):**

1. Load and chunk the PDF → quality checks → normalize text and deduplicate
2. Define the taxonomy and measure keyword coverage on the corpus
3. Auto-label (weak supervision); inspect confidence and ambiguity
4. Simulate human review; build `gold_label`; agreement metrics
5. Train the baseline classifier → confusion matrix → error samples → TF–IDF cues

---

## Learning outcomes

By the end of this workshop, learners will be able to:

- Turn a regulatory PDF into **clean, traceable text chunks** suitable for labeling and modeling.
- Design a **taxonomy**, measure **keyword coverage**, and create **weak labels** with clear diagnostics.
- Run a **review workflow** (queue → corrections → `gold_label`) and interpret **agreement** metrics.
- Train and critique a **baseline text classifier** (metrics, confusion matrix, errors, n‑gram cues).
- **Use Ollama end-to-end:** **`ollama pull`**, confirm with **`ollama list`**, align tags in the notebook, and call the **local** API from Section B and Section C.
- Apply **Section B** patterns (labeling, QA, extraction, summarization, …) with a **local** LLM.
- Run **Section C** English→Korean translation with a pulled **`OLLAMA_TRANSLATION_MODEL`**, preview results, and export **UTF‑8 `.txt`** and **PDF**.


---

# Section A — Classical labeling pipeline (no LLM)

This section runs **end-to-end** from the IGF Code PDF through chunking, normalization, taxonomy-based weak labels, simulated human review, a **TF–IDF + logistic regression** baseline, and evaluation (**Steps 0–6**).

**Format:** use the **markdown** cells for concepts and the following **code** cells for short, runnable steps (avoid single oversized code blocks).

**Prerequisite:** place `data/IGF Code (124Pages).pdf` next to this notebook and run cells **in order** through Section A before Section B (Section B reuses `df`, `taxonomy`, and optionally trained `clf`).

**Section C** also needs `df` from Section A and **`ollama_chat`** / **`OLLAMA_HOST`** from Section B setup (**B0.1–B0.4**), unless you paste those helpers into Section C.

## Step 0 - Environment setup

Run the next cell once. It installs required libraries:

- `pandas`, `numpy`: data processing
- `scikit-learn`: baseline classifier and metrics
- `pypdf`: extract text from the IGF Code PDF
- `tqdm`: optional progress bars

Place the PDF next to this notebook at **`data/IGF Code (124Pages).pdf`** (same path as in this module).

> If your environment already has these packages, installation will be very fast.

In [ ]:
# If needed, uncomment and run (add matplotlib before Section C):
# %pip install -q pandas numpy scikit-learn pypdf tqdm matplotlib

In [ ]:
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score

random.seed(42)
np.random.seed(42)

pd.set_option("display.max_colwidth", 140)
print("Libraries imported successfully.")

## Step 1 - Load and chunk the IGF Code PDF

All training rows in this workshop come from **`data/IGF Code (124Pages).pdf`** (International Code of Safety for Ships using Gases or other Low-flashpoint Fuels).

We:

1. Open the PDF with `pypdf` and extract text page by page.
2. Split each page into **paragraph chunks** (blank-line boundaries), then merge tiny fragments so each row is a usable text unit for classification.
3. Build a dataframe with `inquiry_text` (the chunk), `source_page`, and `chunk_id` for traceability.

> PDF text extraction is imperfect (headers, footers, hyphenation). Real projects often add OCR cleanup or source XML when available.

**Workflow:** (1) define helper functions → (2) extract all pages into a dataframe.

### Step 1a — Helper functions (path + chunking)

`resolve_pdf_path()` searches under `data/` for the IGF PDF. `split_page_into_chunks()` splits page text on blank lines, merges tiny fragments, and caps chunk length so each row is usable for classification.

In [ ]:
def resolve_pdf_path() -> Path:
    """Find PDF relative to current working directory (run notebook from Module 4 folder)."""
    candidates = [
        Path("data") / "IGF Code (124Pages).pdf",
        Path.cwd() / "data" / "IGF Code (124Pages).pdf",
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(
        "Could not find 'data/IGF Code (124Pages).pdf'. "
        "Open Jupyter from the Module 4 folder or set the path manually."
    )


def split_page_into_chunks(
    text: str,
    page_num: int,
    min_chars: int = 80,
    max_chars: int = 2200,
) -> list[dict]:
    """Split raw page text into paragraph-like chunks with length bounds."""
    if not text or not str(text).strip():
        return []
    parts = re.split(r"\n\s*\n+", str(text).strip())
    merged: list[str] = []
    buf = ""
    for p in parts:
        p = re.sub(r"[ \t]+", " ", p.replace("\n", " ")).strip()
        if not p:
            continue
        if len(buf) + len(p) + 1 < min_chars:
            buf = (buf + " " + p).strip() if buf else p
            continue
        if buf:
            merged.append(buf)
            buf = ""
        merged.append(p)
    if buf:
        merged.append(buf)

    rows: list[dict] = []
    for i, block in enumerate(merged):
        if len(block) < min_chars:
            continue
        start = 0
        part_i = 0
        while start < len(block):
            piece = block[start : start + max_chars].strip()
            if len(piece) >= min_chars:
                rows.append(
                    {
                        "source_page": page_num,
                        "chunk_id": f"p{page_num:04d}_b{i:03d}_s{part_i:02d}",
                        "inquiry_text": piece,
                    }
                )
            start += max_chars
            part_i += 1
    return rows


print("resolve_pdf_path and split_page_into_chunks defined.")

### Step 1b — Extract every page and build `df`

Iterate pages with `PdfReader`, extract text, chunk each page, then assemble **`df`**. Optionally subsample with **`MAX_ROWS`** for faster workshops.

In [ ]:
pdf_path = resolve_pdf_path()
print("Using PDF:", pdf_path)

reader = PdfReader(str(pdf_path))
all_rows: list[dict] = []
for idx, page in enumerate(reader.pages, start=1):
    try:
        raw = page.extract_text() or ""
    except Exception as e:
        raw = ""
        print(f"Warning: page {idx} extract failed: {e}")
    all_rows.extend(split_page_into_chunks(raw, idx))

df = pd.DataFrame(all_rows)
if df.empty:
    raise RuntimeError("No text extracted from PDF. Check file path and whether the PDF is text-based.")

MAX_ROWS = 4000
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
else:
    df = df.reset_index(drop=True)

df["source_label_name"] = "igf_code_pdf"

print("Pages:", len(reader.pages), "| Chunks (rows):", len(df))
df[["chunk_id", "source_page", "inquiry_text"]].head(5)

### Step 1c — Exploratory quality checks on extracted chunks

Before labeling, inspect **length**, **page coverage**, and **basic text hygiene**. Regulatory PDFs often produce noisy fragments (page numbers alone, very short blocks that slipped through, uneven paragraph breaks).

We add `_char_len` and `_word_est` for analysis only. Split into two code steps: **length summary** first, then **per-page + short-chunk** views.

#### Step 1c.1 — Length and word-count summary

Copy `inquiry_text` to `inquiry_text_raw` (before Step 1d cleaning) and compute basic length statistics.

In [ ]:
df["inquiry_text_raw"] = df["inquiry_text"].astype(str)
df["_char_len"] = df["inquiry_text_raw"].str.len()
df["_word_est"] = df["inquiry_text_raw"].str.split().str.len()

print("Rows:", len(df))
print("Unique pages:", df["source_page"].nunique())
print("\nCharacter length — describe:")
display(df["_char_len"].describe().to_frame("chars"))

#### Step 1c.2 — Chunks per page and suspiciously short rows

PDF extraction often yields **stubs** (page numbers, running headers). Inspect where chunks concentrate and which rows fall below a length threshold.

In [ ]:
print("Chunks per page (top 8 by count):")
display(df.groupby("source_page").size().sort_values(ascending=False).head(8).to_frame("n_chunks"))

short_thr = 120
short_mask = df["_char_len"] < short_thr
print(f"\nChunks shorter than {short_thr} chars: {short_mask.sum()} ({100 * short_mask.mean():.1f}%)")
df.loc[short_mask, ["chunk_id", "source_page", "_char_len", "inquiry_text_raw"]].head(8)

### Step 1d — Text normalization, noise reduction, and deduplication

**Normalization** reduces variance that hurts both weak labeling and TF-IDF models (line-break hyphens, repeated whitespace, soft hyphen characters).

**Deduplication** removes identical chunk text that appears on multiple pages or after PDF extraction quirks. We keep the **first** occurrence by `chunk_id` order.

After this step, `inquiry_text` is the **canonical** text used for auto-labeling and training. The original string remains in `inquiry_text_raw`.

**Sub-steps:** (1) cleaning function → (2) apply clean + deduplicate.

#### Step 1d.1 — Cleaning function

Remove soft hyphens, reconnect words broken across lines, and normalize whitespace.

In [ ]:
def clean_regulatory_text(s: str) -> str:
    s = str(s)
    s = s.replace("\u00ad", "")
    s = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


print("clean_regulatory_text defined.")

#### Step 1d.2 — Apply cleaning & deduplicate

Map cleaning onto `inquiry_text`, fingerprint normalized text, drop duplicate fingerprints (keep first row).

In [ ]:
before = len(df)
df["inquiry_text"] = df["inquiry_text_raw"].map(clean_regulatory_text)


def fingerprint(t: str) -> str:
    return re.sub(r"\s+", " ", str(t).lower().strip())


df["_fp"] = df["inquiry_text"].map(fingerprint)
dup_mask = df.duplicated(subset=["_fp"], keep="first")
n_dup = int(dup_mask.sum())
df = df.loc[~dup_mask].reset_index(drop=True)
df = df.drop(columns=["_fp"])

print(f"Deduplicated: removed {n_dup} ({n_dup / max(before, 1):.1%} of {before}). Rows now: {len(df)}")

df["_char_len"] = df["inquiry_text"].str.len()
df["_word_est"] = df["inquiry_text"].str.split().str.len()
df[["chunk_id", "source_page", "_char_len", "inquiry_text"]].head(5)

## Step 2 - Taxonomy design for IGF Code text classification

A good taxonomy should be:

- **Mutually exclusive** where possible (regulatory text often overlaps; document borderline rules in guidelines).
- **Collectively exhaustive** for the document types you care about (navigation, search, compliance QA).
- **Actionable**: each class can map to an owner (design office, shipyard, operator, class society).

### Example taxonomy (aligned with typical IGF Code themes)

We define 6 classes for **paragraph-level** chunks from the PDF:

1. `igf_general_definitions` — scope, definitions, general provisions.
2. `igf_ship_design_arrangement` — ship design, arrangement of spaces, structural boundaries.
3. `igf_fuel_containment_systems` — tanks, piping, pressure relief, ventilation of gas fuel systems.
4. `igf_machinery_bunkering` — machinery using gas fuel, bunkering and transfer operations.
5. `igf_operations_emergency` — operational procedures, gas safety, emergency actions.
6. `igf_personnel_surveys_certificate` — training, surveys, certification, records.

Refine keywords with subject-matter experts; weak labels are only a starting point.

**Workflow:** **2a** define the `taxonomy` dict → **2b** preview it as a table → **2c** measure keyword coverage on the corpus.

#### Step 2a — Define the taxonomy dictionary

Each class has a short **description** (for prompts and docs) and **keywords** used later for weak labeling and coverage checks.

In [ ]:
taxonomy = {
    "igf_general_definitions": {
        "description": "Scope, definitions, general requirements, goals, equivalence.",
        "keywords": [
            "definition",
            "general",
            "application",
            "purpose",
            "goal",
            "equivalent",
            "interpretation",
            "regulation",
            "safety",
        ],
    },
    "igf_ship_design_arrangement": {
        "description": "Ship design, arrangement of spaces, location of machinery and tanks.",
        "keywords": [
            "ship",
            "design",
            "arrangement",
            "space",
            "location",
            "deck",
            "compartment",
            "boundary",
            "collision",
            "accommodation",
        ],
    },
    "igf_fuel_containment_systems": {
        "description": "Fuel containment, storage, piping, pressure relief, ventilation.",
        "keywords": [
            "tank",
            "containment",
            "piping",
            "pipe",
            "valve",
            "pressure",
            "relief",
            "lng",
            "gas fuel",
            "fuel gas",
            "ventilation",
            "cryogenic",
        ],
    },
    "igf_machinery_bunkering": {
        "description": "Machinery using gas fuel, bunkering, transfer, fuel supply systems.",
        "keywords": [
            "machinery",
            "engine",
            "generator",
            "bunker",
            "bunkering",
            "transfer",
            "hose",
            "manifold",
            "fuel supply",
            "dual fuel",
        ],
    },
    "igf_operations_emergency": {
        "description": "Operations, procedures, gas detection, emergency response.",
        "keywords": [
            "operation",
            "procedure",
            "manual",
            "emergency",
            "gas detection",
            "detection",
            "alarm",
            "shutdown",
            "leak",
            "fire",
        ],
    },
    "igf_personnel_surveys_certificate": {
        "description": "Training, familiarization, surveys, certification, maintenance, records.",
        "keywords": [
            "training",
            "personnel",
            "familiarization",
            "survey",
            "certificate",
            "inspection",
            "maintenance",
            "testing",
            "record",
            "documentation",
        ],
    },
}

print("Taxonomy classes:", len(taxonomy))

#### Step 2b — Taxonomy table (quick reference)

Display labels, descriptions, and a few example keywords in one dataframe.

In [ ]:
taxonomy_df = pd.DataFrame([
    {"label": k, "description": v["description"], "keyword_examples": ", ".join(v["keywords"][:5])}
    for k, v in taxonomy.items()
])
taxonomy_df

### Step 2c — Taxonomy keyword coverage on the corpus

Weak labels depend on **keyword recall**: if a class’s keywords rarely appear in the PDF chunks, the auto-labeler will under-use that class.

Run **2c.1** (hit counter) then **2c.2** (aggregate per label). Use this to refine keywords or add domain terms (e.g. **CNG**, **LPG**, **fuel cell**).

#### Step 2c.1 — Count keyword hits in a chunk

Normalize whitespace and count how many **distinct** keywords from a list appear as substrings (case-insensitive).

In [ ]:
def count_keyword_hits(text: str, keywords: list[str]) -> int:
    t = re.sub(r"\s+", " ", str(text).lower().strip())
    return sum(1 for kw in keywords if kw.lower() in t)


print("count_keyword_hits defined.")

#### Step 2c.2 — Coverage table per taxonomy label

For each label, count chunks with **at least one** keyword hit and summarize corpus share.

In [ ]:
rows_cov = []
for label, info in taxonomy.items():
    kws = info["keywords"]
    hits = df["inquiry_text"].map(lambda x, ks=kws: count_keyword_hits(x, ks))
    rows_cov.append(
        {
            "label": label,
            "chunks_with_any_keyword": int((hits > 0).sum()),
            "pct_corpus": float((hits > 0).mean()),
            "median_hits_when_present": float(hits[hits > 0].median()) if (hits > 0).any() else 0.0,
        }
    )

coverage_df = pd.DataFrame(rows_cov).sort_values("chunks_with_any_keyword", ascending=False)
coverage_df

## Step 3 - Prompt-based auto-labeling (weak supervision)

In production, prompt-based auto-labeling can use an LLM API with a fixed taxonomy and few-shot examples.

In this workshop, we simulate prompt-based labeling with **keyword scoring** over each IGF-themed class so learners can run offline and inspect decisions.

### Prompt template idea

> "Given a paragraph from the IGF Code, assign exactly one label from the taxonomy; prefer the most specific technical theme."

The auto-labeler scores labels by substring keyword matches (case-insensitive) and picks the best label.

This is a **weak label** stage. Always validate critical compliance use cases with experts.

**Workflow:** **3a** define `normalize_text` + `prompt_based_auto_label` → **3b** apply to `df`.

#### Step 3a — Weak labeler (functions)

`prompt_based_auto_label` scores each taxonomy class by **keyword substring** matches (after normalization) and returns label, a heuristic confidence, and per-class scores.

In [ ]:
def normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def prompt_based_auto_label(text: str, taxonomy_dict: dict, default_label: str = "igf_general_definitions"):
    """
    Simulates prompt-based labeling by scoring keyword matches per taxonomy class.
    Returns: (predicted_label, confidence_score, score_breakdown)
    """
    text_n = normalize_text(text)
    scores = {}

    for label, info in taxonomy_dict.items():
        keywords = info["keywords"]
        score = 0
        for kw in keywords:
            if kw in text_n:
                score += 1
        scores[label] = score

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    if best_score == 0:
        return default_label, 0.25, scores

    sorted_scores = sorted(scores.values(), reverse=True)
    second = sorted_scores[1] if len(sorted_scores) > 1 else 0
    confidence = min(0.99, 0.5 + 0.2 * (best_score - second) + 0.05 * best_score)
    confidence = float(max(0.3, confidence))

    return best_label, confidence, scores


print("normalize_text and prompt_based_auto_label defined.")

#### Step 3b — Apply weak labels to all chunks

Populate `auto_label`, `auto_confidence`, and `score_breakdown` on `df`.

In [ ]:
auto_results = df["inquiry_text"].apply(lambda x: prompt_based_auto_label(x, taxonomy))
df["auto_label"] = auto_results.apply(lambda x: x[0])
df["auto_confidence"] = auto_results.apply(lambda x: x[1])
df["score_breakdown"] = auto_results.apply(lambda x: x[2])

print("Auto-labeling complete.")
df[["inquiry_text", "auto_label", "auto_confidence"]].head(10)

### Step 3c — Diagnostics after auto-labeling

Use **distribution** and **confidence** to see whether the weak labeler is collapsing into one class or leaving many rows uncertain.

We also list **ambiguous** rows where the top two keyword scores are tied or very close (optional review set beyond low-confidence sorting).

**Sub-steps:** **3c.1** distribution & confidence → **3c.2** ambiguity (`score_gap`).

#### Step 3c.1 — Label distribution and confidence

Inspect class counts and the spread of `auto_confidence`.

In [ ]:
print("Auto-label distribution:")
display(df["auto_label"].value_counts().to_frame("n"))

print("\nAuto-confidence describe:")
display(df["auto_confidence"].describe().to_frame("confidence"))

#### Step 3c.2 — Ambiguity via score gap

When the top two class scores are close, the weak label is less trustworthy—flag those rows for review.

In [ ]:
def score_gap(breakdown: dict) -> float:
    vals = sorted(breakdown.values(), reverse=True)
    if len(vals) < 2:
        return float("inf")
    return float(vals[0] - vals[1])


df["_score_gap"] = df["score_breakdown"].map(score_gap)
ambig_thr = 1
ambig = df.loc[df["_score_gap"] <= ambig_thr, ["chunk_id", "source_page", "inquiry_text", "auto_label", "auto_confidence", "score_breakdown"]]
print(f"Ambiguous chunks (top-2 scores within {ambig_thr} hit): {len(ambig)}")
ambig.head(10)

## Step 4 - Labeling training data (IGF Code project)

Now we turn weak labels into better training labels.

Typical workflow:

1. Use auto-labeling to pre-label many PDF chunks quickly.
2. Prioritize **low-confidence** rows (and boundary pages) for expert review.
3. Correct labels; keep `chunk_id` and `source_page` for audit trails.
4. Build final `gold_label` for training or for retrieval metadata.

Below, we prepare a **review queue** from low-confidence samples.

In [ ]:
# Build review queue (focus on uncertain labels)
review_queue = (
    df.sort_values("auto_confidence", ascending=True)
      .loc[:, ["chunk_id", "source_page", "inquiry_text", "auto_label", "auto_confidence", "score_breakdown"]]
      .head(40)
      .reset_index()
      .rename(columns={"index": "row_id"})
)

review_queue.head(12)

### Manual review instruction

In a real workshop session:

- Export `review_queue` to CSV.
- Ask annotators to add `human_label` and `comment`.
- Merge reviewed labels back.

For this notebook demo, we create a simulated reviewed set so the pipeline is runnable end-to-end.

In [ ]:
# Simulate human corrections on a subset of uncertain cases
label_list = list(taxonomy.keys())

sim_review = review_queue[["row_id"]].copy()
if sim_review.empty:
    raise RuntimeError("review_queue is empty; check auto-labeling output.")
# 70% keep model label, 30% corrected (simulation)
sim_review["human_label"] = review_queue["auto_label"]
flip_mask = np.random.rand(len(sim_review)) < 0.30
sim_review.loc[flip_mask, "human_label"] = np.random.choice(label_list, size=flip_mask.sum())

# Merge to main dataframe
review_map = sim_review.set_index("row_id")["human_label"].to_dict()
df["human_label"] = df.index.map(review_map)

# Final label for training: human label overrides auto label when available
df["gold_label"] = df["human_label"].fillna(df["auto_label"])

print("Gold label distribution:")
df["gold_label"].value_counts()

### Step 4b — Agreement between auto-labels and simulated human labels (review subset)

On the rows that entered the review queue, compare **`auto_label`** vs **`human_label`**. In production you would compute the same metrics for real annotators (inter-annotator agreement) and for **auto vs gold** after adjudication.

We report **accuracy** on reviewed rows and **Cohen's kappa** (chance-corrected agreement). Kappa is only meaningful when there is variability in both raters' labels.

In [ ]:
reviewed = df.loc[df["human_label"].notna(), ["auto_label", "human_label"]].copy()
if len(reviewed) == 0:
    print("No human_label values found. Run the simulation cell above first.")
else:
    acc_rev = (reviewed["auto_label"] == reviewed["human_label"]).mean()
    print(f"Reviewed rows: {len(reviewed)} | Auto vs human accuracy: {acc_rev:.3f}")
    try:
        kappa = cohen_kappa_score(reviewed["human_label"], reviewed["auto_label"])
        print(f"Cohen kappa (human vs auto): {kappa:.3f}")
    except ValueError as e:
        print("Kappa not computed:", e)

    # Confusion on reviewed subset only
    labels_r = sorted(set(reviewed["auto_label"]) | set(reviewed["human_label"]))
    cm_r = confusion_matrix(reviewed["human_label"], reviewed["auto_label"], labels=labels_r)
    display(pd.DataFrame(cm_r, index=[f"human_{x}" for x in labels_r], columns=[f"auto_{x}" for x in labels_r]))

## Step 5 — Baseline classifier: train and evaluate

Run the substeps **in order** (each depends on the previous):

- **5a** Training readiness (class balance, label noise proxy)
- **5b** Train/test split (**5b.1**) then TF–IDF + logistic regression (**5b.2**); classification report on the held-out test set
- **5c** Confusion matrix on the held-out test set
- **5d** Sample misclassified chunks (qualitative error analysis)
- **5e** High-weight n-grams per class (linear model interpretability)

> Later, you can replace this block with transformer fine-tuning for higher accuracy.

### Step 5a — Training readiness (class balance and leakage checks)

Before fitting a model:

- Confirm every **gold** class has enough rows for a hold-out split (especially after deduplication).
- Check for **diagnostic columns** you do not want to leak into features (here we only use `inquiry_text` for `X`, so metadata columns are safe if unused).
- Optionally compare a **gold-only** split vs training on **auto labels only** as an ablation (exercise).

The next cell prints class counts and warns if any class has very low support.

In [ ]:
vc_gold = df["gold_label"].value_counts().sort_values(ascending=False)
print("Gold_label counts:")
display(vc_gold.to_frame("n"))

min_cls = int(vc_gold.min())
n_cls = int(vc_gold.shape[0])
print(f"\nClasses: {n_cls} | Smallest class count: {min_cls}")
if min_cls < 5:
    print("Warning: at least one class has < 5 samples; metrics will be noisy and stratify may be unstable.")

# Optional: quick view of label noise proxy (auto vs gold) on full dataframe
noise_proxy = (df["auto_label"] != df["gold_label"]).mean()
print(f"\nShare of rows where auto_label != gold_label (includes non-reviewed rows): {noise_proxy:.3f}")

### Step 5b — Fit the baseline classifier

We map **`inquiry_text` → `gold_label`**, reserve **20%** for testing, then fit a `Pipeline` with `TfidfVectorizer` (word **unigrams and bigrams**) and `LogisticRegression` with `class_weight="balanced"`.

**Sub-steps:** **5b.1** train/test split → **5b.2** build pipeline, `fit`, `predict`, and print the report. This stores `X_train`, `X_test`, `y_train`, `y_test`, `clf`, and `y_pred` for Steps 5c–5e.

#### Step 5b.1 — Train / test split

Stratify when every class has enough support; otherwise fall back to a simple random split.

In [ ]:
X = df["inquiry_text"]
y = df["gold_label"]

vc = y.value_counts()
can_stratify = vc.min() >= 2 and vc.shape[0] > 1
if can_stratify:
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print("Train:", len(X_train), "| Test:", len(X_test))

#### Step 5b.2 — TF–IDF + logistic regression pipeline

Fit on the training split and evaluate accuracy plus per-class metrics on the held-out test set.

In [ ]:
clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=30000)),
    ("model", LogisticRegression(max_iter=400, class_weight="balanced")),
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

### Step 5c — Confusion matrix (held-out test set)

Rows are **true** `gold_label`, columns are **predicted** labels. Use this to see systematic confusions between IGF themes (e.g. operations vs machinery) before reading individual errors.

In [ ]:
labels_sorted = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels_sorted], columns=[f"pred_{l}" for l in labels_sorted])
cm_df

### Step 5d — Error analysis sample (misclassified test chunks)

Reading a handful of **false predictions** is one of the fastest ways to improve labels, taxonomy boundaries, and text cleaning rules.

The next cell prints a few `(y_true, y_pred)` pairs from the held-out test set.

In [ ]:
err_mask = y_pred != y_test.values
err_n = int(err_mask.sum())
print(f"Test errors: {err_n} / {len(y_test)} ({err_n / max(len(y_test), 1):.1%})")

errs = (
    pd.DataFrame(
        {
            "inquiry_text": X_test.values,
            "y_true": y_test.values,
            "y_pred": y_pred,
        }
    )
    .loc[err_mask]
    .head(10)
)
errs

### Step 5e — Interpretability (high-weight TF–IDF n-grams)

Logistic regression on TF–IDF is not a black box: we can inspect **largest positive weights** per class for unigrams/bigrams. This helps validate whether the model focuses on sensible maritime / IGF terms—or on spurious PDF artifacts (page headers, repeated boilerplate).

> Run this **after** Steps 5c–5d so you can connect n-grams to confusion patterns and concrete misclassification examples.

**Sub-steps:** **5e.1** helper → **5e.2** deep dive on one class → **5e.3** top n-gram per class (coarse signature).

#### Step 5e.1 — Extract top positive weights

Given the fitted pipeline and a class name, return the highest-weight TF–IDF features for that class’s one-vs-rest logistic head.

In [ ]:
def top_tfidf_features(pipe: Pipeline, class_name: str, top_k: int = 12):
    model: LogisticRegression = pipe.named_steps["model"]
    vec: TfidfVectorizer = pipe.named_steps["tfidf"]
    feats = np.array(vec.get_feature_names_out())
    if class_name not in model.classes_:
        raise ValueError(f"Unknown class: {class_name}")
    j = list(model.classes_).index(class_name)
    w = model.coef_[j]
    idx = np.argsort(w)[::-1][:top_k]
    return pd.DataFrame({"ngram": feats[idx], "weight": w[idx]}).reset_index(drop=True)


print("top_tfidf_features defined.")

#### Step 5e.2 — Inspect one class

Change `example_class` to another label from `gold_label` to compare strongest n-grams.

In [ ]:
example_class = sorted(y.unique())[0]
print("Example class:", example_class)
display(top_tfidf_features(clf, example_class, top_k=15))

#### Step 5e.3 — One top n-gram per class (rough signature)

Not a full interpretation—just a **sanity check** that each class’s strongest feature looks plausible.

In [ ]:
rows = []
for c in sorted(y.unique()):
    tt = top_tfidf_features(clf, c, top_k=1)
    rows.append({"class": c, "top_ngram": tt.loc[0, "ngram"], "weight": float(tt.loc[0, "weight"])})

pd.DataFrame(rows)

## Step 6 - Interpret results and iterate

You already followed an evaluation sequence: **metrics → confusion matrix → error rows → n-gram weights**. Use those artifacts together—not in isolation—when deciding what to change next.

If model quality is lower than target, improve in this order:

1. **Data & extraction quality** (chunking, normalization, dedupe, PDF noise).
2. **Taxonomy quality**: clarify overlapping definitions.
3. **Label quality**: increase human review on uncertain/conflicting samples.
4. **Data coverage**: collect or generate more examples for minority classes.
5. **Feature/model upgrades**: domain lexicons, transformer embeddings, or fine-tuning.

---

## Suggested exercises for learners

1. Map taxonomy labels to **actual IGF Code chapter/section numbers** and add regex hints.
2. Replace keyword auto-labeling with an **LLM API** prompt + JSON-only output schema.
3. Compare metrics **before vs after** simulated human corrections on the same held-out split.
4. Build **active learning**: rank chunks by classifier entropy or margin, label top-K, retrain.

---

## Optional: save Section A outputs

Use the next cell to export your labeled dataset for reuse or for Section B after a kernel restart (`pd.read_csv`).

In [ ]:
output_path = "igf_code_labeled_training_chunks.csv"
export_cols = [
    "chunk_id",
    "source_page",
    "inquiry_text",
    "inquiry_text_raw",
    "auto_label",
    "auto_confidence",
    "human_label",
    "gold_label",
]
export_df = df[[c for c in export_cols if c in df.columns]].copy()
export_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

---

# Section B — Local LLM workflows (Ollama + pulled models)

This section shows **practical patterns** for using a **local** large language model—labeling, QA, extraction, summarization—via **Ollama's HTTP API**. The default code uses the **`llama3`** tag (Meta Llama 3 family).

**Before Pattern 1:** install Ollama, run **`ollama pull llama3`** (or whatever tag you set in **`OLLAMA_MODEL`**), and confirm it appears in **`ollama list`**. The pull step downloads weights to your machine; without it, API calls will fail.

**Patterns:** each pattern has its **own markdown** (what / why) and **code** (how). Run setup **B0.1 → B0.4** in order, then any pattern; some note **dependencies** (e.g. Pattern 5 uses `zero_shot_classify` from Pattern 1).

**Catalog:**

| # | Pattern | Typical use |
|---|---------|-------------|
| 1 | Zero-shot structured classification | Cheap bulk pre-labels with JSON schema |
| 2 | Few-shot classification | Steer style / domain with a handful of examples |
| 3 | Label + rationale + confidence | Audit trail for reviewers |
| 4 | Top‑k labels + reasons | Ambiguity analysis, active learning queues |
| 5 | Self-consistency (multi-sample voting) | Reduce random reasoning errors |
| 6 | LLM adjudication | Resolve disagreements (e.g. rules vs baseline model) |
| 7 | Batch labeling column | Fill `llm_label` for a sample of `df` |
| 8 | Regulatory obligation extraction | Pull shall/must style duties from text |
| 9 | Chunk summarization | RAG metadata or executive previews |
| 10 | Synthetic study questions | Generate quiz items from each chunk |
| 11 | Contrastive explanation | Why not label B? — for training annotators |
| 12 | Taxonomy critique | Suggest taxonomy tweaks from label confusion summary |
| 13 | Crew checklist rewrite | Turn prose into a short operational checklist |
| 14 | Optional localization | Translate a short summary (e.g. Korean) |
| 15 | Accuracy check | Compare LLM labels vs `gold_label` on a subset |

> **Hardware:** local inference needs RAM/VRAM; start with short prompts and small batches.


## Section B — Prerequisites (Ollama + pulled LLM)

### 1. Install and start Ollama

- Download from **[ollama.com](https://ollama.com)** for Windows / macOS / Linux.
- Keep the **daemon** running (desktop app or run **`ollama serve`**). Default API base: **`http://localhost:11434`**.

### 2. Pull the model weights (required)

Ollama does **not** ship every model pre-installed. You must **download** the weights for each tag you use:

- **Recommended default for this notebook:** open a terminal and run **`ollama pull llama3`**.
  This fetches the **`llama3`** tag (Llama 3) used by **`OLLAMA_MODEL`** in cell **B0.1**.
- **Other tags:** browse [ollama.com/library](https://ollama.com/library), then e.g. **`ollama pull llama3.1:8b`**, **`ollama pull qwen2.5:7b`**, etc. Pulls can take time and **several gigabytes** of disk space.

**Check installation:** **`ollama list`** — your tag must appear before you run Pattern cells.
**Quick chat test:** **`ollama run llama3`** (substitute your tag), ask one question, exit.

If the notebook errors with **connection refused**, Ollama is not running. If it says the **model is missing**, run **`ollama pull <exact-tag>`** again.

### 3. Notebook data

Complete **Section A** so `df` and `taxonomy` exist, **or** load the CSV exported from Section A (uncomment the line in cell **B0.1**).

### 4. Run setup cells

Run **B0.1 → B0.4** in order before Pattern 1.

### 5. Match the tag in code

Set **`OLLAMA_MODEL`** in cell **B0.1** to the **same string** as in **`ollama list`** (e.g. `llama3`, `llama3.1:8b`). Changing the tag in code **without** pulling that tag first will fail at runtime.


## Section B — Setup (run **B0.1 → B0.4** in order)

**Before B0.1:** you should already have run **`ollama pull`** for the tag you will set as **`OLLAMA_MODEL`** (default `llama3`). Confirm with **`ollama list`**.

1. **B0.1** Imports, **`OLLAMA_HOST` / `OLLAMA_MODEL`**, and guard that `df` + `taxonomy` exist (or load CSV).
2. **B0.2** `ollama_chat` — HTTP POST to **`/api/chat`** on your local Ollama server.
3. **B0.3** `extract_json_object` — parse JSON from model text (including fenced code blocks).
4. **B0.4** `taxonomy_prompt_block` + **`SAMPLE_TEXT`** for the patterns below.


### B0.1 — Imports, endpoint, and dataframe guard

**Model tag:** **`OLLAMA_MODEL`** must match a name from **`ollama list`** (after **`ollama pull <tag>`**). Default **`llama3`** corresponds to **`ollama pull llama3`**.

Later patterns use **`time.sleep`** between calls; **`json`** / **`urllib`** call Ollama. Uncomment the CSV line if you restarted the kernel after Section A.


In [ ]:
import json
import re
import time
import urllib.error
import urllib.request

OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "llama3"

# Optional: reload dataframe if you restarted the kernel after Section A
# df = pd.read_csv("igf_code_labeled_training_chunks.csv")

if "df" not in globals() or "taxonomy" not in globals():
    raise RuntimeError("Run Section A through taxonomy + dataframe creation first (or uncomment CSV load above).")

print("B0.1 OK — OLLAMA_HOST:", OLLAMA_HOST, "| model:", OLLAMA_MODEL)

### B0.2 — `ollama_chat`

Non-streaming chat completion; raises a clear error if the daemon is not reachable.

In [ ]:
def ollama_chat(messages, model=OLLAMA_MODEL, temperature=0.2, timeout=180):
    """Chat completion via Ollama /api/chat (non-streaming)."""
    payload = {"model": model, "messages": messages, "stream": False, "options": {"temperature": temperature}}
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        f"{OLLAMA_HOST}/api/chat",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            body = json.loads(resp.read().decode("utf-8"))
    except urllib.error.URLError as e:
        raise RuntimeError(
            f"Cannot reach Ollama at {OLLAMA_HOST}. Start Ollama and run: ollama pull {model}"
        ) from e
    return body["message"]["content"]


print("ollama_chat defined.")

### B0.3 — `extract_json_object`

Strips optional ```json fences and parses the first `{ ... }` block—patterns below expect structured output.

In [ ]:
def extract_json_object(text: str) -> dict:
    """Best-effort parse of first JSON object in model output."""
    text = text.strip()
    if "```" in text:
        fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text, re.I)
        if fence:
            text = fence.group(1).strip()
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        raise ValueError("No JSON object found in model output")
    return json.loads(m.group(0))


print("extract_json_object defined.")

### B0.4 — Taxonomy prompt block + `SAMPLE_TEXT`

`taxonomy_prompt_block()` formats Section A’s `taxonomy` for system/user prompts. `SAMPLE_TEXT` is the first chunk (truncated) for quick experiments.

In [ ]:
def taxonomy_prompt_block() -> str:
    lines = []
    for k, v in taxonomy.items():
        lines.append(f"- `{k}`: {v['description']}")
    return "\n".join(lines)


SAMPLE_TEXT = df["inquiry_text"].iloc[0][:2800]
print("Ollama helpers ready. SAMPLE_TEXT chars:", len(SAMPLE_TEXT), "— continue with Pattern 1, 2, …")

### Pattern 1 — Zero-shot structured classification

Run Section B setup **B0.1 → B0.4** first (`ollama_chat`, `extract_json_object`, `taxonomy_prompt_block`, `SAMPLE_TEXT`).

Provide only the **taxonomy** (from `taxonomy_prompt_block()`) and the **paragraph**. The model returns **one** JSON object: `label`, `confidence`, `reason`.

Best for: fast bulk pre-labeling when you trust the schema and want minimal prompt engineering.

In [ ]:
def zero_shot_classify(chunk: str) -> dict:
    sys = "You classify IGF Code style paragraphs. Reply with ONLY valid JSON, no markdown."
    user = (
        f"Taxonomy:\n{taxonomy_prompt_block()}\n\n"
        f"Paragraph:\n{chunk}\n\n"
        'Return JSON: {"label":"<exactly_one_taxonomy_key>","confidence":0.0,"reason":"<short>"}'
    )
    out = ollama_chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.1)
    return extract_json_object(out)


print("Pattern 1:", zero_shot_classify(SAMPLE_TEXT))

### Pattern 2 — Few-shot classification

Prepend **short user/assistant turns** (example → gold JSON) before the real paragraph. Replace placeholders with **real** excerpts from your corpus in production.

*Requires setup **B0.1–B0.4** (same prerequisites as Pattern 1); uses the same JSON schema as Pattern 1.*

In [ ]:
few_examples = [
    {
        "role": "user",
        "content": "Example (ship arrangement): text mentions collision bulkhead location → classify",
    },
    {
        "role": "assistant",
        "content": '{"label":"igf_ship_design_arrangement","confidence":0.86,"reason":"Structural arrangement"}',
    },
]


def few_shot_classify(chunk: str) -> dict:
    sys = "You are an expert maritime classifier. Output ONLY JSON per schema."
    user = (
        f"{taxonomy_prompt_block()}\n\n"
        f"Paragraph:\n{chunk}\n\n"
        'Return JSON: {"label":"...","confidence":0.0,"reason":"..."}'
    )
    msgs = [{"role": "system", "content": sys}, *few_examples, {"role": "user", "content": user}]
    out = ollama_chat(msgs, temperature=0.15)
    return extract_json_object(out)


print("Pattern 2:", few_shot_classify(SAMPLE_TEXT))

### Pattern 3 — Audit-style JSON (label + rationale + confidence)

Ask for extra fields such as **`uncertainty_note`** so reviewers can filter borderline labels.

Best for: compliance workflows where you need a **short audit trail** per chunk.

In [ ]:
def audit_style(chunk: str) -> dict:
    sys = "Return ONLY JSON with keys label, confidence (0-1), reason (<=25 words), uncertainty_note (optional)."
    user = f"Taxonomy:\n{taxonomy_prompt_block()}\n\nText:\n{chunk}"
    out = ollama_chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.15)
    return extract_json_object(out)


print("Pattern 3:", audit_style(SAMPLE_TEXT[:2000]))

### Pattern 4 — Top‑k labels with reasons

Ask for **several** candidate labels ranked with brief **why** text—useful for **ambiguous** regulation language and **active learning** queues.

Parse JSON key `top` as a list of `{label, confidence, why}`.

In [ ]:
def topk_labels(chunk: str, k: int = 3) -> dict:
    sys = (
        "Return ONLY JSON: "
        '{"top":[{"label":str,"confidence":float,"why":str}]} with exactly '
        + str(k)
        + " items."
    )
    user = f"Taxonomy:\n{taxonomy_prompt_block()}\n\nText:\n{chunk}"
    out = ollama_chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.2)
    return extract_json_object(out)


print("Pattern 4:", topk_labels(SAMPLE_TEXT[:2000]))

### Pattern 5 — Self-consistency (majority vote)

Call **`zero_shot_classify`** multiple times (same chunk) and take a **majority vote** on `label` to reduce one-off reasoning mistakes.

*Requires: Pattern 1 code cell (`zero_shot_classify`).*

In [ ]:
from collections import Counter


def self_consistency_vote(chunk: str, n: int = 3) -> dict:
    votes = []
    for _ in range(n):
        r = zero_shot_classify(chunk)
        votes.append(r["label"])
        time.sleep(0.4)
    winner, cnt = Counter(votes).most_common(1)[0]
    return {"labels": votes, "majority": winner, "agreement": cnt / n}


print("Pattern 5:", self_consistency_vote(SAMPLE_TEXT[:1800], n=3))

### Pattern 6 — LLM adjudication (keyword vs classifier)

When **keyword `auto_label`** and the **Section A baseline `clf`** disagree, ask the LLM to choose the best taxonomy label and justify.

*Requires: Pattern 1 (`zero_shot_classify` optional); **Section A Step 5** must have been run so `clf` exists.*

In [ ]:
def adjudicate(key_pred: str, clf_pred: str, chunk: str) -> dict:
    sys = "You resolve labeling disputes for IGF-style text. Output ONLY JSON."
    user = (
        f"Taxonomy:\n{taxonomy_prompt_block()}\n\n"
        f"Keyword model guessed: {key_pred}\nLinear model guessed: {clf_pred}\n\n"
        f"Paragraph:\n{chunk}\n\n"
        'Return JSON: {"label":"...","confidence":0.0,"reason":"which evidence favored your choice"}'
    )
    out = ollama_chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.2)
    return extract_json_object(out)


if "clf" in globals():
    row = df.iloc[0]
    txt = row["inquiry_text"][:2400]
    print(
        "Pattern 6:",
        adjudicate(row["auto_label"], clf.predict([txt])[0], txt),
    )
else:
    print("Pattern 6 skipped: run Section A Step 5 first so `clf` exists.")

### Pattern 7 — Batch LLM labels (`llm_label` column)

Fill **`llm_label`** for the first *m* dataframe rows using **`zero_shot_classify`**. Keep *m* small; insert **`time.sleep`** between calls to avoid overloading local inference.

*Requires: Pattern 1 (`zero_shot_classify`).*

In [ ]:
def batch_llm_labels(df_in: pd.DataFrame, m: int = 5, sleep_s: float = 0.5) -> pd.DataFrame:
    out = []
    sub = df_in.head(m).copy()
    for _, row in sub.iterrows():
        try:
            js = zero_shot_classify(str(row["inquiry_text"])[:2800])
            out.append(js.get("label"))
        except Exception as e:
            out.append(f"ERROR:{e}")
        time.sleep(sleep_s)
    sub["llm_label"] = out
    return sub


df_llm_sample = batch_llm_labels(df, m=3)
display(df_llm_sample[["chunk_id", "gold_label", "llm_label"]])

### Pattern 8 — Regulatory obligation extraction

Ask the model to list **shall / must / should** style obligations as bullets—useful for **requirement mining** and downstream structured databases.

*Uses only the setup helpers (`ollama_chat`).*

In [ ]:
def extract_obligations(chunk: str) -> str:
    sys = "Extract obligations. Output plain bullet lines starting with '- ', each one concise."
    user = "List regulatory obligations (shall/must/should) implied by:\n" + chunk[:3000]
    return ollama_chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.1)


print("Pattern 8 (snippet):\n", extract_obligations(SAMPLE_TEXT)[:800])

### Pattern 9 — Chunk summarization (RAG metadata)

Produce **one–two sentences** suitable as document/chunk metadata for **retrieval** or catalog screens.

In [ ]:
def rag_summary(chunk: str) -> str:
    sys = "Summarize for metadata indexing. Two sentences max."
    return ollama_chat(
        [{"role": "system", "content": sys}, {"role": "user", "content": chunk[:3200]}],
        temperature=0.2,
    )


print("Pattern 9:", rag_summary(SAMPLE_TEXT))

### Pattern 10 — Synthetic inspection-style question

Ask for **one** exam-style question plus **answer** and a **short citation** from the chunk—useful for training assessments or student quizzes.

*Requires **`extract_json_object`** from setup cell **B0.3** (and **`ollama_chat`** from **B0.2**).*

In [ ]:
def synthetic_question(chunk: str) -> dict:
    sys = "Return ONLY JSON keys: question, answer, citations (short quote from text)."
    user = "From this IGF-style excerpt, write ONE inspection-style question:\n" + chunk[:2800]
    out = ollama_chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.3)
    return extract_json_object(out)


print("Pattern 10:", synthetic_question(SAMPLE_TEXT))

### Pattern 11 — Contrastive explanation (why A vs B)

Given two taxonomy labels, ask why **A** is more appropriate than **B** for this excerpt—good for **annotator training** and error analysis.

In [ ]:
def contrastive(chunk: str, label_a: str, label_b: str) -> str:
    user = (
        f"Why is `{label_a}` more appropriate than `{label_b}` for this text? 4 bullet points.\n\n"
        f"Text:\n{chunk[:2600]}"
    )
    return ollama_chat(
        [{"role": "system", "content": "Be faithful to the excerpt."}, {"role": "user", "content": user}],
        temperature=0.2,
    )


la, lb = list(taxonomy.keys())[0], list(taxonomy.keys())[3]
print("Pattern 11:\n", contrastive(SAMPLE_TEXT, la, lb)[:900])

### Pattern 12 — Taxonomy critique from qualitative notes

Paste **human-readable** confusion patterns (from confusion matrices, tickets, SME interviews). The model suggests **merge / split / rename** moves—not ground truth.

Uses slightly higher **temperature** for brainstorming.

In [ ]:
def critique_taxonomy(notes: str) -> str:
    sys = "You advise on ontology design for maritime regulators."
    user = (
        f"Current labels:\n{taxonomy_prompt_block()}\n\n"
        f"Observed issues / confusion pairs:\n{notes}\n\n"
        "Suggest concrete edits (merge/split/rename), bullet list."
    )
    return ollama_chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.35)


demo_notes = (
    "Annotators confuse igf_fuel_containment_systems vs igf_machinery_bunkering "
    "when both piping and engines appear."
)
print("Pattern 12:\n", critique_taxonomy(demo_notes)[:1200])

### Pattern 13 — Crew checklist rewrite

Turn dense regulatory prose into a **numbered checklist** for training—not a replacement for official SMS/procedures.

*Standalone prompt; no dependency on earlier pattern cells.*

In [ ]:
def crew_checklist(chunk: str) -> str:
    sys = "Rewrite as numbered checklist for ship staff training (concise, actionable)."
    return ollama_chat(
        [{"role": "system", "content": sys}, {"role": "user", "content": chunk[:3200]}],
        temperature=0.25,
    )


print("Pattern 13:\n", crew_checklist(SAMPLE_TEXT)[:900])

### Pattern 14 — Localization (example: Korean)

Translate an English summary into **Korean**—adjust the target language in the system prompt for other locales.

*Requires: Pattern 9 (`rag_summary`) so we translate a short summary rather than the full chunk.*

In [ ]:
def translate_ko(text_en: str) -> str:
    sys = "Translate accurately to Korean. Preserve technical terms where standard."
    return ollama_chat(
        [{"role": "system", "content": sys}, {"role": "user", "content": text_en}],
        temperature=0.15,
    )


try:
    sum_en = rag_summary(SAMPLE_TEXT[:2000])
except NameError as e:
    raise RuntimeError("Run Pattern 9 first to define rag_summary().") from e
print("Pattern 14 (KO):", translate_ko(sum_en)[:500])

### Pattern 15 — LLM vs `gold_label` (subset accuracy)

Compare **`zero_shot_classify`** to **`gold_label`** on the first *n* rows. This is a **rough sanity check** only (small *n*, expensive API calls to self-hosted inference).

*Requires: Pattern 1 (`zero_shot_classify`).*

In [ ]:
def eval_llm_vs_gold(n: int = 5) -> float:
    ok = 0
    used = 0
    for i in range(min(n, len(df))):
        row = df.iloc[i]
        chunk = str(row["inquiry_text"])[:2400]
        gold = row["gold_label"]
        try:
            pred = zero_shot_classify(chunk)["label"]
            used += 1
            if pred == gold:
                ok += 1
        except Exception:
            pass
        time.sleep(0.45)
    return ok / used if used else 0.0


print("Pattern 15 accuracy (small n):", eval_llm_vs_gold(n=5))

## Section B — Wrap-up

- Combine patterns: e.g. **top‑k (4)** → send borderline rows to **human** queue; use **self-consistency (5)** only on those.
- Keep **temperature low** for labeling JSON; raise slightly for brainstorming (**12**, critique).
- Log **prompts + raw outputs** for audit—especially for compliance-facing workflows.
- For production scale, use **batch APIs**, **structured output guards**, and **evaluation harnesses** (hold-out sets, not just LLM-vs-gold on tiny *n*).

---

**Next:** **Section C** — translate English excerpts to Korean (configurable Ollama models), preview in the notebook, and export PDF + text files.

---

# Section C — English → Korean translation & PDF export

This section translates **`inquiry_text`** chunks from Section A (English regulatory prose) to **Korean** using a **local Ollama** model, shows a **preview** in the notebook, and writes:

- a UTF‑8 **`.txt`** file (full translated document)
- a **`.pdf`** file (paginated; uses a **Korean-capable system font** when available)

**Prerequisites:** Section A (`df` with `inquiry_text`) **and** Section B setup **B0.1** (defines **`OLLAMA_HOST`**, **`OLLAMA_MODEL`** / optional CSV load) plus **B0.2** (`ollama_chat`). You do **not** need Patterns 1–15—only those setup cells (or paste equivalent helpers into Section C).

**Ollama model:** set **`OLLAMA_TRANSLATION_MODEL`** to a tag you have **pulled** (`ollama pull <tag>`). Verify with **`ollama list`**. Section C may use a **different** tag than Section B (e.g. a stronger multilingual model for EN→KO).

> Full-document translation can take **a long time**; the code caps chunks / characters per run (`MAX_CHUNKS`, `MAX_CHARS_PER_CHUNK`). Increase after smoke tests.

**Structure:** numbered steps — each step has short **markdown** + **code** (imports → config → translation → fonts → PDF → batch → preview → export).


## Section C — Choosing an Ollama model for Korean

**Workflow:** (1) Pick a tag from the [Ollama library](https://ollama.com/library). (2) **`ollama pull <tag>`** and wait for the download. (3) Confirm **`ollama list`**. (4) Set **`OLLAMA_TRANSLATION_MODEL`** in the Section C config cell to that **exact** tag. (5) Re-run translation cells after any tag change.

Default in code: **`llama3`** (reasonable multilingual behavior). For **English→Korean** you may prefer another tag—**always pull first**:

| Direction | Example Ollama tags (availability varies by registry) | Notes |
|-----------|--------------------------------------------------------|--------|
| Strong multilingual MT | `qwen2.5:7b`, `qwen2.5:14b`, `mistral`, `mistral-nemo` | Often solid EN↔KO for general prose; verify with your domain samples. |
| Updated Llama instruct | `llama3.1:8b`, `llama3.2:3b` | Better instruction-following than bare `llama3`; try if translations omit constraints. |
| Google Gemma 2 | `gemma2:9b`, `gemma2:2b` | Multilingual; lighter weights for laptops. |
| Korean-focused open weights | Community builds (names change): search **Ollama library** for **“Korean”**, **“Ko-”**, **“Open-Ko”**, **“EEVE”**, **“Polyglot”** | If a Korean-tuned Llama/Mistral tag exists in your environment, it is often the best first try for **terminology** and **spacing**. |
| Smaller / faster checks | `llama3.2:1b`, `phi3:mini` | Fast smoke tests; quality may be lower for legal/technical text. |

**Practice:** keep the **same** test chunks, switch only **`OLLAMA_TRANSLATION_MODEL`**, and compare outputs side by side in the saved `.txt` files.

**Font / PDF:** PDF rendering uses **matplotlib** and whatever **Korean font** is installed (e.g. **Malgun Gothic** on Windows, **Noto Sans CJK KR** on many Linux installs). If you see **tofu** (□), install a Korean TTF and register it, or open the `.txt` in a UTF‑8 editor and print to PDF from there.


### Section C — Step 1: Imports & prerequisites

We need **`ollama_chat`** from Section B (same Ollama server) and **`df`** from Section A. **matplotlib** renders the PDF; **`IPython.display`** shows a readable preview in the notebook.

Ensure you have **`ollama pull <tag>`** for whatever you set as **`OLLAMA_TRANSLATION_MODEL`** (see the Section C model table); **`ollama list`** should list that tag.

> Run `%pip install matplotlib` if the import fails.

In [ ]:
# If needed: %pip install -q matplotlib

import textwrap
import time
from pathlib import Path

import matplotlib.font_manager as mfont
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import Markdown, display

if "ollama_chat" not in globals():
    raise RuntimeError(
        "Run Section B cells B0.1 and B0.2 first (Ollama helpers), or define ollama_chat() here."
    )
if "df" not in globals():
    raise RuntimeError("Run Section A to build `df`, or load the exported CSV.")

print("Imports OK — dataframe rows:", len(df))

### Section C — Step 2: Configuration

- **`OLLAMA_TRANSLATION_MODEL`** — Ollama tag for EN→KO (see table above); change and re-run downstream cells to compare models.
- **`TRANSLATE_TEMPERATURE`** — lower ≈ more literal; slightly higher can read more fluent but less tightly anchored.
- **`MAX_CHUNKS` / `MAX_CHARS_PER_CHUNK`** — limits cost and runtime during teaching.
- **`OUT_TXT` / `OUT_PDF`** — output filenames in the notebook folder.

In [ ]:
OLLAMA_TRANSLATION_MODEL = "llama3"  # e.g. "qwen2.5:7b", "llama3.1:8b", "gemma2:9b"
TRANSLATE_TEMPERATURE = 0.15

MAX_CHUNKS = 4
MAX_CHARS_PER_CHUNK = 3200
SLEEP_BETWEEN_CALLS = 0.35

OUT_TXT = Path("igf_code_translated_ko.txt")
OUT_PDF = Path("igf_code_translated_ko.pdf")

print("Model:", OLLAMA_TRANSLATION_MODEL, "| max chunks:", MAX_CHUNKS, "| out:", OUT_TXT.name, "/", OUT_PDF.name)

### Section C — Step 3: Translation helper (`translate_en_to_ko`)

A single **system prompt** asks for maritime-flavored Korean and **output only** the translation. Very long English is split into **segments** (character windows) so each Ollama request stays stable; segments are concatenated with blank lines.

The function calls **`ollama_chat(..., model=OLLAMA_TRANSLATION_MODEL)`** so it respects your chosen tag.

In [ ]:
def translate_en_to_ko(text_en: str, model: str | None = None) -> str:
    """English → Korean using Ollama; defaults to OLLAMA_TRANSLATION_MODEL."""
    text_en = (text_en or "").strip()
    if not text_en:
        return ""
    mdl = model or OLLAMA_TRANSLATION_MODEL
    sys_msg = (
        "You are a professional translator for maritime safety and ship regulations. "
        "Translate the following English into natural Korean. "
        "Keep article/paragraph numbering if present. Preserve technical terms using standard Korean maritime usage where applicable. "
        "Output ONLY the Korean translation—no preface, no quotes, no English commentary."
    )
    segments = []
    step = min(MAX_CHARS_PER_CHUNK, 4500)
    for i in range(0, len(text_en), step):
        part = text_en[i : i + step]
        out = ollama_chat(
            [{"role": "system", "content": sys_msg}, {"role": "user", "content": part}],
            model=mdl,
            temperature=TRANSLATE_TEMPERATURE,
        )
        segments.append(out.strip())
        time.sleep(SLEEP_BETWEEN_CALLS)
    return "\n\n".join(segments)


print("translate_en_to_ko defined.")

### Section C — Step 4: Pick a Korean font for matplotlib (PDF)

Hangul needs a font that contains Korean glyphs. We scan a **priority list** (e.g. Malgun Gothic on Windows, Noto/Nanum on many setups). If none match, PDF generation still runs but may show **□** unless you install a font.

In [ ]:
def pick_korean_matplotlib_font() -> str | None:
    candidates = [
        "Malgun Gothic",
        "NanumGothic",
        "Nanum Gothic",
        "Noto Sans CJK KR",
        "Noto Sans KR",
        "Apple SD Gothic Neo",
        "AppleGothic",
    ]
    available = {f.name for f in mfont.fontManager.ttflist}
    for name in candidates:
        if name in available:
            print("Using PDF font:", name)
            return name
    print("No Korean font found in matplotlib registry — PDF may not render Hangul.")
    return None


print("pick_korean_matplotlib_font defined.")

### Section C — Step 5: Save Korean text as paginated PDF (`save_korean_pdf`)

We wrap lines with **`textwrap`**, pack ~42 lines per **A4** page, and draw each page with **`PdfPages`**. This is simple and dependency-light; for publication-quality layout consider HTML→PDF or dedicated typesetting tools.

*Requires: Steps 1–4 (imports, config, `pick_korean_matplotlib_font`).*

In [ ]:
def save_korean_pdf(text_ko: str, pdf_path: Path, title: str = "IGF Code — Korean (excerpt)") -> None:
    font = pick_korean_matplotlib_font()
    if font is None:
        print(
            "Warning: using DejaVu Sans — Hangul may appear as boxes. Use UTF-8 .txt or install a Korean font."
        )
        font = "DejaVu Sans"

    body = (title + "\n\n" + text_ko).strip()
    lines: list[str] = []
    for para in body.split("\n"):
        if not para.strip():
            lines.append("")
            continue
        lines.extend(textwrap.wrap(para, width=88, replace_whitespace=False) or [para])

    lines_per_page = 42
    pages: list[str] = []
    buf: list[str] = []
    for line in lines:
        buf.append(line)
        if len(buf) >= lines_per_page:
            pages.append("\n".join(buf))
            buf = []
    if buf:
        pages.append("\n".join(buf))

    with PdfPages(pdf_path) as pdf:
        for pg in pages:
            fig, ax = plt.subplots(figsize=(8.27, 11.69))
            ax.set_axis_off()
            ax.text(
                0.06,
                0.96,
                pg,
                transform=ax.transAxes,
                fontsize=9,
                va="top",
                ha="left",
                fontfamily=font,
                wrap=False,
            )
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print("PDF written:", pdf_path.resolve())


print("save_korean_pdf defined.")

### Section C — Step 6: Batch-translate chunks & save `.txt`

Iterate **`df.head(MAX_CHUNKS)`**, prepend a small header (`chunk_id`, page) so merged files stay traceable, call **`translate_en_to_ko`** per row, then write **`igf_code_translated_ko.txt`** (UTF‑8).

*Requires: Steps 2–3 (`MAX_*`, `translate_en_to_ko`).*

In [ ]:
parts_ko: list[str] = []
sub = df.head(MAX_CHUNKS)
for idx, row in sub.iterrows():
    cid = row.get("chunk_id", f"row_{idx}")
    pg = row.get("source_page", "")
    en = str(row["inquiry_text"])[:MAX_CHARS_PER_CHUNK]
    header = f"--- [{cid}] page {pg} ---"
    print("Translating", cid, "…")
    ko_block = translate_en_to_ko(en)
    parts_ko.append(header + "\n" + ko_block)
    time.sleep(SLEEP_BETWEEN_CALLS)

full_ko = "\n\n".join(parts_ko)
OUT_TXT.write_text(full_ko, encoding="utf-8")
print("Saved UTF-8 text:", OUT_TXT.resolve(), "| chars:", len(full_ko))

### Section C — Step 7: Preview in the notebook

Large translations would overwhelm the UI; we **`display(Markdown(...))`** only the first ~6000 characters and mark truncation. Open **`OUT_TXT`** for the full document.

In [ ]:
preview = full_ko[:6000] + (
    "\n\n… [truncated for display] …" if len(full_ko) > 6000 else ""
)
display(Markdown("## Korean preview (excerpt)\n\n" + preview))

### Section C — Step 8: Write the PDF file

Calls **`save_korean_pdf(full_ko, ...)`** so the PDF matches the same Korean text you saved as `.txt`. If Hangul appears as **□**, install a Korean font (see Step 4) or export PDF from the `.txt` using Word/LibreOffice.

*Requires: Steps 5–7 (`save_korean_pdf`, variable `full_ko`).*

In [ ]:
save_korean_pdf(
    full_ko,
    OUT_PDF,
    title=f"IGF Code — Korean translation ({OLLAMA_TRANSLATION_MODEL})",
)

## Section C — End

**Outputs:** `igf_code_translated_ko.txt` (full UTF‑8), `igf_code_translated_ko.pdf` (paginated).

**Tip:** To compare models, change **`OLLAMA_TRANSLATION_MODEL`** in Step 2, then re-run **Steps 6–8** (batch translate → preview → PDF). Optionally rename `OUT_TXT` / `OUT_PDF` in Step 2 so files do not overwrite (e.g. `…_qwen.txt`).

---

**End of notebook** — Sections A (classical pipeline), B (Ollama patterns), C (EN→KO + PDF).